# Credit Card Fraud Detection

## Objective
Build a binary classifier that identifies fraudulent transactions. The primary selection metric is **F1 score** because the data are highly imbalanced; **PR-AUC** is reported as a threshold-independent secondary metric.

This notebook keeps `train.csv` separate from `val.csv` to avoid selecting a model or threshold on the data used for fitting.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score, f1_score, precision_recall_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

from credit_fraud_utils_data import split_features_target
from credit_fraud_utils_eval import evaluate_probabilities, find_best_threshold

RANDOM_STATE = 42
DATA_DIR = Path('data/split')
sns.set_theme(style='whitegrid', context='notebook')

## 1. Load and audit the data
The PCA-transformed `V1`-`V28` fields are anonymized. `Time` and `Amount` are the only directly interpretable predictors.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
validation = pd.read_csv(DATA_DIR / 'val.csv')

audit = pd.DataFrame({
    'rows': [len(train), len(validation)],
    'fraud_cases': [int(train.Class.sum()), int(validation.Class.sum())],
    'fraud_rate': [train.Class.mean(), validation.Class.mean()],
    'missing_values': [int(train.isna().sum().sum()), int(validation.isna().sum().sum())],
}, index=['train', 'validation'])
audit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=train, x='Class', ax=axes[0], palette=['#4C78A8', '#E45756'], hue='Class', legend=False)
axes[0].set_title('Training class distribution')
axes[0].set_xlabel('Class (1 = fraud)')

sns.boxplot(data=train, x='Class', y='Amount', showfliers=False, ax=axes[1], palette=['#4C78A8', '#E45756'], hue='Class', legend=False)
axes[1].set_title('Transaction amount by class')
axes[1].set_xlabel('Class (1 = fraud)')
plt.tight_layout()

## 2. Model candidates
The required baselines are logistic regression, random forest, and a soft-voting ensemble. An extra-trees benchmark is included as an additional sklearn-only tree ensemble. All models are evaluated using the same held-out validation set, and each receives its own F1-optimal threshold.

In [ ]:
X_train, y_train = split_features_target(train)
X_validation, y_validation = split_features_target(validation)

candidates = {
    'Logistic regression': Pipeline([
        ('scaler', RobustScaler()),
        ('classifier', LogisticRegression(class_weight='balanced', max_iter=2000, solver='liblinear', random_state=RANDOM_STATE)),
    ]),
    'Random forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE
    ),
    'Extra trees': ExtraTreesClassifier(
        n_estimators=350, class_weight='balanced', max_features='sqrt', n_jobs=-1, random_state=RANDOM_STATE
    ),
}

results, fitted_models = [], {}
for name, model in candidates.items():
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_validation)[:, 1]
    threshold, _ = find_best_threshold(y_validation.to_numpy(), probabilities)
    metrics = evaluate_probabilities(y_validation.to_numpy(), probabilities, threshold)
    metrics['model'] = name
    results.append(metrics)
    fitted_models[name] = (model, probabilities, threshold)

results_frame = pd.DataFrame(results).sort_values(['f1', 'pr_auc'], ascending=False)
results_frame[['model', 'f1', 'pr_auc', 'precision', 'recall', 'threshold', 'predicted_fraud_count']]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, (_, probabilities, _) in fitted_models.items():
    PrecisionRecallDisplay.from_predictions(y_validation, probabilities, name=name, ax=ax)
ax.set_title('Validation precision-recall curves')
plt.tight_layout()

## 3. Conclusions

- Fraud is rare, so accuracy would be misleading and is deliberately not used for model selection.
- The operating threshold is tuned after fitting rather than fixed at 0.50, which substantially improves the F1 trade-off.
- The final saved artifact should use the model with the highest validation F1 after the required candidate exploration.
- Run `python credit_fraud_train.py --model xgboost` to train the selected gradient-boosting model and write `model.pkl`.